In [1]:
# 하나의 논문 url에서 저자와 저자의 id을 추출하는 코드 

import requests
from bs4 import BeautifulSoup
import re  # 정규표현식 모듈 추가

def get_dbpia_author_ids(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        authors_data = []
        
        # 저자 링크(a 태그)를 모두 찾습니다.
        # 이미지 기준: class="authorName"
        author_links = soup.select('#authorWrap .authorName')
        
        for link in author_links:
            # 1. onclick 속성의 문자열을 그대로 가져옵니다.
            onclick_text = link.get('onclick') 
            # 예: "... 'type_value':'이진희', 'type_id': '433938822' ..."
            
            if onclick_text:
                # 2. 정규표현식으로 데이터 추출
                # 'type_value' 뒤에 오는 따옴표 안의 내용을 찾음
                name_match = re.search(r"'type_value':\s*'([^']*)'", onclick_text)
                # 'type_id' 뒤에 오는 따옴표 안의 내용을 찾음
                id_match = re.search(r"'type_id':\s*'([^']*)'", onclick_text)
                
                # 매칭된 것이 있으면 가져오고, 없으면 None
                type_value = name_match.group(1) if name_match else None
                type_id = id_match.group(1) if id_match else None
                
                if type_value or type_id:
                    authors_data.append({
                        'type_value': type_value, # 저자 이름
                        'type_id': type_id        # 저자 고유 ID
                    })
                    
        return authors_data

    except Exception as e:
        print(f"Error: {e}")
        return []

# --- 실행 예시 ---
target_url = "https://www.dbpia.co.kr/journal/articleDetail?nodeId=NODE10544378"
results = get_dbpia_author_ids(target_url)

for res in results:
    print(f"저자: {res['type_value']}, ID: {res['type_id']}")

저자: 이진희, ID: 433938822
저자: 최정락, ID: 631435050
저자: 구강, ID: 912490882


In [6]:
# 원본 json 파일에서 url만 추출해서 정리

import re
import os

# ==========================================
# [설정] 파일 이름 확인
# ==========================================
INPUT_FILE = '../SSU_Datathon2025_공학분야_62199.json'          # 원본 파일명
OUTPUT_FILE = 'extracted_urls.txt' # 저장할 파일명
# ==========================================

def force_extract_urls():
    print(f"1. '{INPUT_FILE}' 파일을 강제 분석 모드로 엽니다...")
    
    if not os.path.exists(INPUT_FILE):
        print("오류: 파일을 찾을 수 없습니다.")
        return

    # 1. 파일 전체를 텍스트로 읽어옵니다.
    # (인코딩 에러 방지를 위해 utf-8 시도 후 실패하면 cp949로 재시도)
    content = ""
    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            content = f.read()
    except UnicodeDecodeError:
        print(" -> utf-8 읽기 실패, cp949(윈도우 기본)로 다시 시도합니다...")
        try:
            with open(INPUT_FILE, 'r', encoding='cp949') as f:
                content = f.read()
        except Exception as e:
            print(f"파일 읽기 치명적 오류: {e}")
            return

    # 2. 정규표현식으로 "NODE_LINK":"(주소)" 패턴을 모두 찾습니다.
    # 설명: "NODE_LINK" 뒤에 콜론(:)이 오고, 따옴표(") 안에 있는 내용을 잡습니다.
    pattern = r'"NODE_LINK"\s*:\s*"([^"]+)"'
    
    extracted_urls = re.findall(pattern, content)
    count = len(extracted_urls)

    print(f"-> 총 {count}개의 URL을 발견했습니다!")

    # 3. 결과 저장
    if count > 0:
        print(f"2. '{OUTPUT_FILE}' 파일에 저장합니다...")
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            for url in extracted_urls:
                f.write(url + '\n')
        print("완료! 파일을 확인해보세요.")
    else:
        print("여전히 0개입니다. 파일 내용에 'NODE_LINK' 글자가 있는지 메모장으로 확인해보세요.")

if __name__ == "__main__":
    force_extract_urls()

1. '../SSU_Datathon2025_공학분야_62199.json' 파일을 강제 분석 모드로 엽니다...
-> 총 62199개의 URL을 발견했습니다!
2. 'extracted_urls.txt' 파일에 저장합니다...
완료! 파일을 확인해보세요.


In [7]:
# url이 정리된 텍스트 파일을 가지고 이름과 id을 추출하는 크롤링 코드

import requests
from bs4 import BeautifulSoup
import re
import time
import random
import csv
import os

# ==========================================
# [설정] 파일 이름 설정
# ==========================================
INPUT_TXT_FILE = 'extracted_urls.txt'      # URL이 들어있는 텍스트 파일
OUTPUT_CSV_FILE = 'dbpia_final_results.csv' # 결과가 저장될 엑셀(CSV) 파일
# ==========================================

def get_authors_from_url(url):
    """
    특정 URL에 접속하여 저자 이름과 ID를 리스트로 반환합니다.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Referer': 'https://www.dbpia.co.kr/'
    }
    
    results = []
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            print(f"  -> 접속 실패 (Status: {response.status_code})")
            return []

        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 저자 태그 찾기 (아까 확인한 id="authorWrap" 내부의 class="authorName")
        author_links = soup.select('#authorWrap .authorName')
        
        for link in author_links:
            onclick_text = link.get('onclick')
            if onclick_text:
                # 정규표현식으로 자바스크립트 코드 내 변수 추출
                # 예: 'type_value':'홍길동', 'type_id': '12345'
                name_match = re.search(r"'type_value':\s*'([^']*)'", onclick_text)
                id_match = re.search(r"'type_id':\s*'([^']*)'", onclick_text)
                
                name = name_match.group(1) if name_match else ""
                auth_id = id_match.group(1) if id_match else ""
                
                if name or auth_id:
                    results.append({
                        'author_name': name,
                        'author_id': auth_id
                    })
                    
        return results

    except Exception as e:
        print(f"  -> 에러 발생: {e}")
        return []

def main():
    print("=== DBpia 저자 정보 추출기 가동 ===")
    
    # 1. URL 파일 읽기
    if not os.path.exists(INPUT_TXT_FILE):
        print(f"오류: '{INPUT_TXT_FILE}' 파일이 없습니다. 이전 단계를 먼저 실행해주세요.")
        return

    with open(INPUT_TXT_FILE, 'r', encoding='utf-8') as f:
        # 공백 제거하고 빈 줄은 제외하여 리스트로 만듦
        urls = [line.strip() for line in f if line.strip()]
    
    total_count = len(urls)
    print(f"-> 총 {total_count}개의 URL을 확인했습니다.\n")
    
    # 2. 크롤링 및 CSV 저장
    # 'utf-8-sig'를 사용해야 엑셀에서 한글이 깨지지 않습니다.
    with open(OUTPUT_CSV_FILE, 'w', newline='', encoding='utf-8-sig') as csvfile:
        writer = csv.writer(csvfile)
        # 헤더(제목) 쓰기
        writer.writerow(['Source URL', 'Author Name', 'Author ID'])
        
        print(f"2. 크롤링 시작! 결과는 '{OUTPUT_CSV_FILE}'에 저장됩니다.")
        
        for idx, url in enumerate(urls, 1):
            print(f"[{idx}/{total_count}] 처리 중... ", end="")
            
            authors = get_authors_from_url(url)
            
            if authors:
                for author in authors:
                    writer.writerow([url, author['author_name'], author['author_id']])
                print(f"완료 ({len(authors)}명 추출)")
            else:
                print("저자 정보 없음 (또는 실패)")
            
            # [중요] 서버 차단 방지 (1~3초 랜덤 대기)
            time.sleep(random.uniform(1, 3))

    print(f"\n=== 작업 완료! '{OUTPUT_CSV_FILE}' 파일을 확인하세요. ===")

if __name__ == "__main__":
    main()

=== DBpia 저자 정보 추출기 가동 ===
-> 총 62199개의 URL을 확인했습니다.

2. 크롤링 시작! 결과는 'dbpia_final_results.csv'에 저장됩니다.
[1/62199] 처리 중... 완료 (1명 추출)
[2/62199] 처리 중... 완료 (5명 추출)
[3/62199] 처리 중... 완료 (4명 추출)
[4/62199] 처리 중... 완료 (5명 추출)
[5/62199] 처리 중... 완료 (7명 추출)
[6/62199] 처리 중... 완료 (5명 추출)
[7/62199] 처리 중... 완료 (1명 추출)
[8/62199] 처리 중... 완료 (3명 추출)
[9/62199] 처리 중... 완료 (1명 추출)
[10/62199] 처리 중... 완료 (5명 추출)
[11/62199] 처리 중... 완료 (5명 추출)
[12/62199] 처리 중... 완료 (3명 추출)
[13/62199] 처리 중... 완료 (3명 추출)
[14/62199] 처리 중... 완료 (5명 추출)
[15/62199] 처리 중... 완료 (2명 추출)
[16/62199] 처리 중... 완료 (2명 추출)
[17/62199] 처리 중... 완료 (5명 추출)
[18/62199] 처리 중... 완료 (1명 추출)
[19/62199] 처리 중... 완료 (2명 추출)
[20/62199] 처리 중... 완료 (4명 추출)
[21/62199] 처리 중... 완료 (6명 추출)
[22/62199] 처리 중... 완료 (3명 추출)
[23/62199] 처리 중... 저자 정보 없음 (또는 실패)
[24/62199] 처리 중... 완료 (1명 추출)
[25/62199] 처리 중... 완료 (2명 추출)
[26/62199] 처리 중... 완료 (5명 추출)
[27/62199] 처리 중... 완료 (1명 추출)
[28/62199] 처리 중... 완료 (1명 추출)
[29/62199] 처리 중... 완료 (3명 추출)
[30/62199] 처리 중... 완료 (2명 추출)


KeyboardInterrupt: 

In [1]:
# 좀 더 빨라진 크롤링 코드

import requests
from bs4 import BeautifulSoup
import re
import time
import random
import csv
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm  # 진행바 라이브러리 추가

# ==========================================
# [설정]
# ==========================================
INPUT_TXT_FILE = 'extracted_urls.txt'
OUTPUT_CSV_FILE = 'dbpia_extract_authors.csv'
MAX_WORKERS = 10  # 속도를 위해 10명 유지 (에러나면 5로 줄이세요)
# ==========================================

def get_authors_from_url(url):
    """개별 URL 작업을 처리하는 일꾼 함수"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Referer': 'https://www.dbpia.co.kr/'
    }
    
    results = []
    try:
        # 차단 방지용 미세 딜레이
        time.sleep(random.uniform(0.5, 1.5))
        
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            return [] # 실패 시 빈 리스트 반환

        soup = BeautifulSoup(response.text, 'html.parser')
        author_links = soup.select('#authorWrap .authorName')
        
        for link in author_links:
            onclick_text = link.get('onclick')
            if onclick_text:
                name_match = re.search(r"'type_value':\s*'([^']*)'", onclick_text)
                id_match = re.search(r"'type_id':\s*'([^']*)'", onclick_text)
                
                name = name_match.group(1) if name_match else ""
                auth_id = id_match.group(1) if id_match else ""
                
                if name or auth_id:
                    results.append({
                        'source_url': url,
                        'author_name': name,
                        'author_id': auth_id
                    })
        return results

    except Exception:
        return [] # 에러 나도 멈추지 않고 그냥 넘어감

def main():
    print(f"=== 대량 크롤링 시작 (총 {MAX_WORKERS} 스레드) ===")
    
    if not os.path.exists(INPUT_TXT_FILE):
        print("URL 파일이 없습니다.")
        return

    with open(INPUT_TXT_FILE, 'r', encoding='utf-8') as f:
        urls = [line.strip() for line in f if line.strip()]
    
    total_count = len(urls)
    print(f"-> 총 {total_count}개의 논문을 처리합니다.\n")

    # CSV 파일 열기
    with open(OUTPUT_CSV_FILE, 'w', newline='', encoding='utf-8-sig') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['Source URL', 'Author Name', 'Author ID'])
        
        # ThreadPoolExecutor로 병렬 처리
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            # 모든 작업을 등록
            future_to_url = {executor.submit(get_authors_from_url, url): url for url in urls}
            
            # [핵심] tqdm으로 감싸서 진행 바 생성
            # total=total_count: 전체 개수 알려줌
            # desc="크롤링 중": 진행 바 왼쪽에 뜰 텍스트
            # unit="논문": 단위 표시
            for future in tqdm(as_completed(future_to_url), total=total_count, desc="진행률", unit="건"):
                try:
                    data = future.result()
                    
                    if data:
                        for row in data:
                            writer.writerow([row['source_url'], row['author_name'], row['author_id']])
                            
                except Exception as e:
                    # 진행 바 깨짐을 방지하기 위해 에러 출력은 가급적 자제하거나
                    # tqdm.write()를 사용해야 함
                    pass

    print(f"\n=== 작업 완료! '{OUTPUT_CSV_FILE}' 저장됨 ===")

if __name__ == "__main__":
    main()

=== 대량 크롤링 시작 (총 10 스레드) ===
-> 총 62199개의 논문을 처리합니다.



진행률: 100%|██████████| 62199/62199 [3:24:33<00:00,  5.07건/s]    


=== 작업 완료! 'dbpia_extract_authors.csv' 저장됨 ===
